In [1]:
import json  # 1. json 모듈 임포트
from w1thermsensor import W1ThermSensor, Sensor
import time

sensor_map = {
    "0625659202d1": "S1",
    "40ba008750ff": "S2",
    "0625659b7f3e": "S3",
    "062565416895": "S4",
}

sensors = [sensor for sensor in W1ThermSensor.get_available_sensors() if sensor.type == Sensor.DS18B20]

if len(sensors) > 0:
    try:
        while True:
            # 2. JSON으로 변환할 빈 딕셔너리 생성
            sensor_data = {}
            
            for sensor in sensors:
                sensor_name = sensor_map.get(sensor.id, f"unknown({sensor.id})")
                temperature_c = None
                
                for attempt in range(2):
                    try:
                        temperature_c = sensor.get_temperature()
                        break
                    except Exception:
                        time.sleep(0.1)
                
                # 3. 측정된 값을 딕셔너리에 저장 (추후 데이터 처리를 위해 float 숫자형 유지)
                if temperature_c is not None:
                    sensor_data[sensor_name] = round(temperature_c, 1)
                else:
                    sensor_data[sensor_name] = "error"
            
            # 4. 딕셔너리를 JSON 포맷 문자열로 변환하여 한 줄로 출력
            json_output = json.dumps(sensor_data)
            print(json_output)
            
            time.sleep(2)

    except KeyboardInterrupt:
        print("\n모니터링 종료")
else:
    print(json.dumps({"error": "DS18B20 센서를 찾을 수 없습니다."}))

{"S2": 22.8, "S1": 24.1, "S4": 23.8, "S3": 23.9}
{"S2": 22.8, "S1": 24.2, "S4": 23.8, "S3": 23.9}

모니터링 종료


In [ ]:
import time
import json  # JSON 변환을 위한 모듈 추가
import board
import busio
import adafruit_tca9548a
import adafruit_adxl34x

# ------------------------------------------------
# 1. I2C 및 TCA9548A 멀티플렉서 초기화
# ------------------------------------------------
i2c = busio.I2C(board.SCL, board.SDA)
tca = adafruit_tca9548a.TCA9548A(i2c)

# ------------------------------------------------
# 2. 4개의 ADXL345 센서 초기화 및 매핑
# ------------------------------------------------
target_channels = [0, 1, 2, 3] 
sensors = {}

for ch in target_channels:
    try:
        sensor = adafruit_adxl34x.ADXL345(tca[ch])
        sensors[ch] = sensor
    except Exception as e:
        # JSON 포맷이 깨지지 않도록 초기화 실패 메시지는 생략하거나 별도 처리 가능
        pass 

time.sleep(1)

# ------------------------------------------------
# 3. 실시간 데이터 수신 루프 (JSON + 영점 보정)
# ------------------------------------------------
offsets = {
    0: (0.7, 0.5, -0.7),
    1: (0.2, 0.4, -0.2),
    2: (0.6, 0.4, -0.6),
    3: (0.6, 0.2, 0.7)
}

if not sensors:
    print(json.dumps({"error": "연결된 센서가 하나도 없습니다."}))
else:
    try:
        while True:
            # 1. JSON으로 변환할 빈 딕셔너리 생성
            sensor_data = {}
            
            for ch, sensor in sensors.items():
                channel_name = f"CH{ch}"
                try:
                    raw_x, raw_y, raw_z = sensor.acceleration
                    off_x, off_y, off_z = offsets.get(ch, (0.0, 0.0, 0.0))
                    
                    cal_x = raw_x - off_x
                    cal_y = raw_y - off_y
                    cal_z = raw_z - off_z
                    
                    # 2. 보정된 X, Y, Z 값을 소수점 첫째 자리까지 딕셔너리에 저장
                    sensor_data[channel_name] = {
                        "X": round(cal_x, 1),
                        "Y": round(cal_y, 1),
                        "Z": round(cal_z, 1)
                    }
                except OSError:
                    sensor_data[channel_name] = "error"
            
            # 3. 딕셔너리를 JSON 문자열로 변환 (sort_keys=True로 CH0~CH3 정렬)
            json_output = json.dumps(sensor_data, sort_keys=True)
            print(json_output)
            
            time.sleep(0.5) 
            
    except KeyboardInterrupt:
        pass # 종료 시 불필요한 텍스트가 출력되지 않도록 처리